# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset on second primary colorectal cancer using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This analysis uses the Croissant schema provided at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the FAIR² Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Each entity in the schema is uniquely identified by its `@id`.

We will enumerate all record sets available.

In [ ]:
# List all record sets by @id with their fields
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print(f"Found {len(record_sets)} record set(s):")
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
        fields = list(rs.fields)
        print("  Fields:")
        for field in fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")

## 3. Data Extraction
Load data from one or more record sets into DataFrames. All references use Croissant schema `@id` fields for clarity and reproducibility.

Let's extract all available records for each record set.

In [ ]:
# Prepare DataFrames for all available record sets
dataframes = {}
record_sets = list(dataset.record_sets)
record_set_ids = [rs.id for rs in record_sets]
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Loaded {len(df)} records from RecordSet: {rs.name} (@id: {rs.id})")
        else:
            print(f"No records found for RecordSet: {rs.name} (@id: {rs.id})")
    except Exception as e:
        print(f"Could not load records for {rs.name} (@id: {rs.id}): {e}")

# Display first table's columns and a sample if any tables exist
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns in first loaded RecordSet (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No data loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
We will demonstrate cleaning, filtering, and transforming data from one record set (if available).

Replace the field `@id`s below as appropriate for your analysis. In this demo, we attempt to select a numeric field and a grouping field based on type hints.

In [ ]:
# Sample EDA: pick the first record set with numeric fields, else demonstrate on available data
import numpy as np

if dataframes:
    # Select the first record set containing at least one numeric field
    found = False
    for rs in dataset.record_sets:
        df = dataframes.get(rs.id)
        if df is not None and not df.empty:
            numeric_field = None
            group_field = None
            # Find first numeric-type field
            for field in rs.fields:
                if field.data_type in ('Number', 'Float', 'Integer') and field.id in df.columns:
                    numeric_field = field.id
                    break
            # Find a categorical/text field
            for field in rs.fields:
                if field.data_type in ('Text', 'String') and field.id in df.columns:
                    group_field = field.id
                    break
            if numeric_field is None:
                # Fallback: any numeric-looking column
                for col in df.select_dtypes(include=np.number).columns:
                    numeric_field = col
                    break
            if group_field is None:
                # Fallback: any object/string column
                for col in df.select_dtypes(include='object').columns:
                    group_field = col
                    break
            if numeric_field is not None:
                # Ready for EDA
                found = True
                selected_df = df.copy()
                print(f"Using RecordSet: {rs.name} (@id: {rs.id})")
                print(f"Numeric field for filtering/normalization: {numeric_field}")
                threshold = selected_df[numeric_field].quantile(0.8) if pd.api.types.is_numeric_dtype(selected_df[numeric_field]) else None
                if threshold is not None:
                    filtered_df = selected_df[selected_df[numeric_field] > threshold]
                    print(f"Filtered records with {numeric_field} > {threshold} (approx. top 20%):\n", filtered_df[[numeric_field]].head())
                    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
                    print(f"\nNormalized {numeric_field} for filtered records (z-score):\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
                    if group_field in filtered_df.columns:
                        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
                        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
                        display(grouped_df.head())
                else:
                    print(f"No numeric data found to filter/normalize for field: {numeric_field}")
                break
    if not found:
        print("No suitable numeric fields found for EDA.")
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Based on available numeric and group fields, let's visualize distributions and group comparisons using Matplotlib and Seaborn.

Adjust field `@id` and record set ids as appropriate for more in-depth plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=10, color='skyblue')
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in filtered_df.columns and filtered_df[group_field].nunique() < 30:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Not enough numeric/groupable data for visualization.")

## 6. Conclusion
This notebook demonstrated fetching metadata and records from a FAIR² Croissant dataset, exploring record sets and fields using their unique `@id`s, and performing basic analysis and visualization with `mlcroissant` and pandas.

- All interactions referenced schema entities by their `@id`.
- For exploration and visualization, we recommended customizing field selection based on the actual record set structure (review cell outputs above).

Further steps may involve:
- Deeper feature engineering and selection.
- Domain-specific outlier handling.
- Applying predictive analytics or ML models using your selected features.
